---
# Part 1 - Results from the GS's are Analyzed 
---
- purpose of the Part 1 of thenotebook is to analyze the metrics from an initial exploratory gridsearch 
- after this largetr GS, i run a more concentrated GS with less parameters , parameters are eliminated based on the visual analysis below 
- more robust methods may be used but this works for now 

In [15]:
import pandas as pd
import pickle
import random
import numpy as np
import os
import itertools
from joblib import Parallel, delayed , parallel_backend
from collections import defaultdict
import math
import torch.nn as nn
import json

from Equations_Run_Combo_V_2 import *



import pickle
with open('/Users/cs/Desktop/LSTM_ETF/short_dfs.pkl', 'rb') as f:
    loaded_dfs = pickle.load(f)

with open("/Users/cs/Desktop/LSTM_ETF/lagged_cache.pkl", "rb") as f:
    lagged_cache = pickle.load(f)


#/home/charifslmn/

---
# Part 1 - GS Results Exploration
---

In [16]:

for i in range(1):  
    with open(f'/Users/cs/Desktop/GS_21_01_to_22_12_chunk_1_32_10percentPOS_Vset_UCO.json', 'r') as f:
        results = json.load(f)

len(results)


4096

In [17]:
import json
import numpy as np
import copy


scored_results = []
for entry in results:
    avg = entry['cv_sets'].get('overall_metrics', {})
    acc = avg.get('accuracy') or 0.0
    prec_up = avg.get('precision_up') or 0.0
    recall_up = avg.get('recall_up') or 0.0

    total_score = acc  # or add prec_up, etc.
    
    if recall_up > 10 and prec_up > 65:

        scored_results.append((total_score, entry))

# Step 2: Sort by score descending
scored_results.sort(reverse=True, key=lambda x: x[0])


top = scored_results[:70]

for i, j in top:
    print(j["combo_number"], '<--->' , j["cv_sets"]["overall_metrics"] , '<---->' , j["parameters"] )

# Step 4: Extract the parameter combos
combos = [j["parameters"] for i, j in top]

print(f"Total Combos Selected: {len(top)}")

1733 <---> {'accuracy': 79.16666666666666, 'precision_up': 87.5, 'recall_up': 63.63636363636363, 'precision_down': 75.0, 'recall_down': 92.3076923076923} <----> {'binary_0_1_cutoff_ret_rate_percentage': 0.1, 'learning_rate': 0.005, 'num_epochs': 150, 'batch_size': 30, 'use_bidirectional': False, 'lag': 2, 'input_size': 12, 'hidden_size': 12, 'num_layers': 4, 'use_monthly_dfs_only': True, 'use_binary_0_1_retRate': False, 'use_custom_loss_function_BCE_THRESH': True, 'use_custom_loss_function_BCE_THRESH_AND_SEVERITY': False, 'use_LOW_weights_for_BCE_custom_loss': True, 'pred_threshold_sigmoid01_up': None, 'use_binary_neg1_1': False, 'use_ret_rate': False, 'use_print_acc': False, 'use_dropout': False, 'use_class_weighting': False, 'is_deterministic': True, 'seed_num': 42, 'use_existing_lagged_data': True, 'use_dynamic_weights': False, 'use_binary_0_1_retRate_custom_neg': False, 'use_binary_0_1_retRate_custom_pos': True, 'POS_weight_multiplier': 1, 'use_rolling_fixed_train_size': False, 'us

In [18]:
### pickle combos 

with open('/Users/cs/Desktop/LSTM_ETF/DIST_DISC_comboc_GS_21_01_to_22_12_chunk_1_32_10percentPOS_Vset_UCO.pkl', 'wb') as f:
    pickle.dump(combos, f)

---
# Part 2: GS Best Results Distribution Discovery 
---

In [19]:
from Equations_Run_Combo_V_2 import *
from Equations_Ensembles_Dist import *

In [ ]:

#### ----------------------                 RUN COMBOS
with parallel_backend("loky", n_jobs=3):
    results = Parallel()(
        delayed(distribution_discovery)(c, combo_index = i , number_of_seeds = 70)
        
        for i, c in enumerate(combos)
    )




KeyboardInterrupt: 

In [ ]:

class NpEncoder(json.JSONEncoder):
    def default(self, obj):
        if isinstance(obj, numpy.integer):
            return int(obj)
        elif isinstance(obj, numpy.ndarray):
            return obj.tolist()
        elif isinstance(obj, pd.DatetimeIndex):
            return obj.tolist()
        elif isinstance(obj, pd.Timestamp):
            return obj.isoformat()
        else:
            return super(NpEncoder, self).default(obj)


file = f"70_models_GS_21_01_to_22_12_DIST_Discovery_10percentPOS_Vset_HOD.json"

with open(file, "w") as f_json:
    json.dump(results, f_json, cls=NpEncoder, indent=2)